# 02 — Feature Engineering

**Goal:** Transform clean data into model-ready features.

**Input:** `data/processed/movies_full_298k.csv` (from `01_data_pipeline.ipynb`)

**Outputs:**

| File | Rows | Description |
|------|------|-------------|
| `movies_full_wide.csv` | ~298k | All movies with IMDb features (TMDB features where available) |
| `movies_rich_wide.csv` | ~39k | Only movies with complete TMDB data (includes embeddings) |
| `pca_transformer.pkl` | — | Fitted PCA transformer for plot embeddings |

---
## 1. Setup

In [ ]:
import pickle
import sys

import pandas as pd
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

# Display settings
pd.set_option("display.max_columns", None)

# Paths
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

print("Setup complete!")

---
## 2. Load Data

In [ ]:
# Load the full dataset (298k movies with TMDB data where available)
df = pd.read_csv(INPUT_DIR / "movies_full_298k.csv")

print(f"Loaded {len(df):,} movies")
print(f"Columns: {list(df.columns)}")

# Check TMDB coverage
has_tmdb = (df["overview"].fillna("").astype(str) != "").sum()
print(f"\nMovies with TMDB data: {has_tmdb:,} ({has_tmdb/len(df)*100:.1f}%)")

df.head(3)

---
## 3. IMDb Features

Features derived from IMDb data (available for all 298k movies).

| Source | Features |
|--------|----------|
| IMDb | `movie_age`, `decade`, `runtimeMinutes_capped`, `log_numVotes`, `hit`, `Genre_*` (one-hot) |
| TMDB | `log_budget`, `log_revenue`, `pca_0` to `pca_19` (plot embeddings) |

> **Note:** Title features (`title_has_colon`, title embedding PCA) were tested in the
> model v6 trial and found to have no measurable impact on prediction accuracy.
> They are excluded from the v5 pipeline.

### 3.1 Temporal Features

In [ ]:
# Movie age (years since release)
CURRENT_YEAR = 2026
df["movie_age"] = CURRENT_YEAR - df["startYear"]

# Decade (e.g., 1995 -> 1990)
df["decade"] = (df["startYear"] // 10 * 10).astype("Int64")

print("Temporal features created:")
print(f"  movie_age: min={df['movie_age'].min()}, max={df['movie_age'].max()}")
print(f"  decade: {sorted(df['decade'].dropna().unique())}")

### 3.2 Runtime Capping

In [ ]:
# Cap runtime at 300 minutes (5 hours) to handle outliers
RUNTIME_CAP = 300

outliers = (df["runtimeMinutes"] > RUNTIME_CAP).sum()
df["runtimeMinutes_capped"] = df["runtimeMinutes"].clip(upper=RUNTIME_CAP)

print(f"Runtime capped at {RUNTIME_CAP} minutes")
print(f"  Movies affected: {outliers:,}")

### 3.3 Popularity Features

In [ ]:
# Log transform numVotes (highly skewed)
df["log_numVotes"] = np.log1p(df["numVotes"])

# Hit flag: top 20% by votes
threshold_80 = df["numVotes"].quantile(0.80)
df["hit"] = (df["numVotes"] >= threshold_80).astype(int)

print(f"Popularity features created:")
print(f"  log_numVotes: range [{df['log_numVotes'].min():.2f}, {df['log_numVotes'].max():.2f}]")
print(f"  hit threshold: {threshold_80:,.0f} votes")
print(f"  hit movies: {df['hit'].sum():,} ({df['hit'].mean()*100:.1f}%)")

### 3.4 Genre Encoding

In [ ]:
# Count genres per movie
df["genre_count"] = df["genres"].str.split(",").str.len()

# Get all genres and their counts
all_genres = df["genres"].str.split(",").explode()
genre_counts = all_genres.value_counts()

# Keep genres with >= 1000 occurrences
MIN_GENRE_COUNT = 1000
valid_genres = genre_counts[genre_counts >= MIN_GENRE_COUNT].index.tolist()

print(f"Total unique genres: {len(genre_counts)}")
print(f"Genres with >= {MIN_GENRE_COUNT} occurrences: {len(valid_genres)}")
print(f"Dropped: {sorted(set(genre_counts.index) - set(valid_genres))}")

In [ ]:
# One-hot encode valid genres
for genre in valid_genres:
    df[f"Genre_{genre}"] = df["genres"].str.contains(genre, regex=False).astype(int)

genre_cols = [col for col in df.columns if col.startswith("Genre_")]
print(f"Created {len(genre_cols)} genre columns")

---
## 4. TMDB Features

Features derived from TMDB data (only available for ~39k movies).

### 4.1 Budget & Revenue

In [ ]:
# Log transform budget and revenue
df["log_budget"] = np.log1p(df["budget"])
df["log_revenue"] = np.log1p(df["revenue"])

print("Budget/Revenue features created:")
print(f"  Movies with budget > 0: {(df['budget'] > 0).sum():,}")
print(f"  Movies with revenue > 0: {(df['revenue'] > 0).sum():,}")

### 4.2 Plot Embeddings

Generate embeddings from movie overviews using SentenceTransformer, then reduce with PCA.

**Important:** We fit PCA only on movies with real overviews (~39k), then transform all 298k. This ensures PCA learns meaningful plot patterns, not just "has overview vs doesn't".

In [ ]:
# Load model
print("Loading SentenceTransformer model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

In [ ]:
# Generate embeddings for ALL movies (empty overview = zero vector)
print("Generating embeddings for all movies...")
print("(Movies without overview will get zero vectors)")

overviews = df["overview"].fillna("").astype(str).tolist()
embeddings = model.encode(overviews, show_progress_bar=True, batch_size=64)

print(f"\nEmbeddings shape: {embeddings.shape}")

In [ ]:
# Apply PCA to reduce dimensions (384 -> 20)
# FIT only on movies with real overviews, TRANSFORM all movies
N_COMPONENTS = 20

# Identify movies with real overviews
has_overview = df["overview"].fillna("").astype(str) != ""
rich_indices = df[has_overview].index
print(f"Fitting PCA on {len(rich_indices):,} movies with real overviews...")

# Fit PCA on rich subset only
pca = PCA(n_components=N_COMPONENTS)
pca.fit(embeddings[rich_indices])
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

# Transform ALL embeddings using the fitted PCA
print(f"Transforming all {len(embeddings):,} movies...")
embeddings_pca = pca.transform(embeddings)

# Add PCA columns to dataframe
pca_cols = [f"pca_{i}" for i in range(N_COMPONENTS)]
for i, col in enumerate(pca_cols):
    df[col] = embeddings_pca[:, i]

print(f"Added {len(pca_cols)} PCA columns")

---
## 5. Create Wide Tables

In [ ]:
# Define feature columns
imdb_features = [
    "movie_age", "decade", "runtimeMinutes_capped",
    "log_numVotes", "hit", "genre_count", "isAdult"
] + genre_cols

tmdb_features = [
    "log_budget", "log_revenue", "has_budget", "has_revenue"
] + pca_cols

# Columns to keep
id_cols = ["tconst"]
target_col = ["averageRating"]
all_features = imdb_features + tmdb_features

print(f"ID columns: {len(id_cols)}")
print(f"Target: {target_col}")
print(f"IMDb features: {len(imdb_features)}")
print(f"TMDB features: {len(tmdb_features)}")
print(f"Total features: {len(all_features)}")

In [ ]:
# Create full wide table (298k movies)
df_full_wide = df[id_cols + target_col + all_features].copy()

# Create rich wide table (only movies with TMDB data)
has_overview = df["overview"].fillna("").astype(str) != ""
df_rich_wide = df[has_overview][id_cols + target_col + all_features].copy()

print(f"movies_full_wide: {len(df_full_wide):,} rows x {len(df_full_wide.columns)} cols")
print(f"movies_rich_wide: {len(df_rich_wide):,} rows x {len(df_rich_wide.columns)} cols")

---
## 6. Export

In [ ]:
# Export both wide tables
full_path = OUTPUT_DIR / "movies_full_wide.csv"
rich_path = OUTPUT_DIR / "movies_rich_wide.csv"

df_full_wide.to_csv(full_path, index=False)
print(f"✓ {full_path.name}: {len(df_full_wide):,} rows")

df_rich_wide.to_csv(rich_path, index=False)
print(f"✓ {rich_path.name}: {len(df_rich_wide):,} rows")

print(f"\nDone! Files saved to: {OUTPUT_DIR}")

---
## 7. Export PCA Transformer

Save the fitted PCA transformer for use in the prediction pipeline.

In [ ]:
# Create models directory if it doesn't exist
MODELS_DIR.mkdir(exist_ok=True)

# Save the fitted PCA transformer
pca_path = MODELS_DIR / "pca_transformer.pkl"
with open(pca_path, "wb") as f:
    pickle.dump(pca, f)

print(f"✓ PCA transformer saved to: {pca_path}")
print(f"  Components: {pca.n_components_}")
print(f"  Explained variance: {pca.explained_variance_ratio_.sum():.2%}")

---
## Summary

| Output File | Rows | Columns | Description |
|-------------|------|---------|-------------|
| `data/processed/movies_full_wide.csv` | ~298k | 55 | All movies, 53 features + id + target |
| `data/processed/movies_rich_wide.csv` | ~39k | 55 | TMDB subset, same 53 features |
| `models/pca_transformer.pkl` | — | — | Fitted PCA (20 components) for plot embeddings |

**Features (53 total):**
- IMDb core (7): `movie_age`, `decade`, `runtimeMinutes_capped`, `log_numVotes`, `hit`, `genre_count`, `isAdult`
- Genres (22): one-hot encoded
- TMDB financials (4): `log_budget`, `log_revenue`, `has_budget`, `has_revenue`
- Plot embeddings (20): `pca_0` through `pca_19`

Model v5 uses **49 of these 53 features** (drops `log_numVotes`, `hit`, `log_revenue`, `has_revenue` — not available pre-release).

**Next** → `03_model_training.ipynb` trains and evaluates model v5.